# Bài học 7: Model và Cách Lựa Chọn

**Thời gian:** ~40 phút | **Chi phí:** $0 (không gọi API)

## Tại Sao Cần Nhiều Model Khác Nhau?

Giống như có nhiều loại xe cho nhiều nhu cầu khác nhau, có nhiều AI model cho nhiều tác vụ khác nhau.

Ví dụ với xe hơi:
- **Toyota Camry** — đáng tin cậy, giá phải chăng, phù hợp đi lại hàng ngày
- **Ferrari** — nhanh, đắt, dành cho lúc cần hiệu năng đỉnh cao
- **Xe bán tải** — không nhanh, nhưng chở được hàng nặng

AI model cũng vậy:
- **Claude Sonnet** — nhanh, giá hợp lý, rất mạnh với tools và structured output
- **Claude Opus** — chậm hơn, đắt hơn, tốt nhất cho suy luận phức tạp
- **Grok-4** — viết văn dài rất tốt, có những thế mạnh riêng

Không có model nào là "tốt nhất" — tất cả phụ thuộc vào tác vụ.

## Đánh Đổi Giữa Tốc Độ / Chi Phí / Chất Lượng

Mỗi model cân bằng ba yếu tố. Bạn không thể có cả ba ở mức tối đa.

| Model | Tốc độ | Chi phí | Phù hợp nhất cho |
|-------|--------|---------|-------------------|
| Claude Haiku | Rất nhanh | Rất rẻ | Tác vụ đơn giản, phân loại |
| Claude Sonnet | Nhanh | Vừa phải | Tools, structured output, phân tích |
| Claude Opus | Chậm | Đắt | Suy luận phức tạp, vai trò leader |
| Grok-4 | Trung bình | Vừa phải | Viết văn dài, nội dung sáng tạo |

In [ ]:
# Hãy ước tính chi phí pipeline cho mỗi bài viết

# Chi phí xấp xỉ trên 1M token (tính đến 2025)
models = {
    "Claude Sonnet": {"input": 3.00, "output": 15.00},
    "Claude Opus": {"input": 15.00, "output": 75.00},
    "Grok-4": {"input": 3.00, "output": 15.00},
}

# Lượng token sử dụng trong pipeline (xấp xỉ)
pipeline_steps = [
    {"step": "Research Agent", "model": "Claude Sonnet", "input_tokens": 2000, "output_tokens": 1500},
    {"step": "Outline Agent", "model": "Claude Sonnet", "input_tokens": 3000, "output_tokens": 1000},
    {"step": "Writer Agent", "model": "Grok-4", "input_tokens": 3000, "output_tokens": 3500},
    {"step": "Image Agent", "model": "Claude Sonnet", "input_tokens": 5000, "output_tokens": 500},
]

total_cost = 0
print("Chi phí chi tiết cho mỗi bài viết:\n")
for step in pipeline_steps:
    model = models[step["model"]]
    input_cost = step["input_tokens"] / 1_000_000 * model["input"]
    output_cost = step["output_tokens"] / 1_000_000 * model["output"]
    step_cost = input_cost + output_cost
    total_cost += step_cost
    print(f"  {step['step']:20s} ({step['model']:14s}): ${step_cost:.4f}")

print(f"\n  {'TỔNG CỘNG':20s}                  : ${total_cost:.4f}")
print(f"\n  Cho 100 bài viết: ${total_cost * 100:.2f}")
print(f"  Cho 1000 bài viết: ${total_cost * 1000:.2f}")

## Giải Thích Các Lựa Chọn Model Của Chúng Ta

Đây là các quyết định thực tế chúng ta đã đưa ra trong `builders.py` — file định nghĩa tất cả các agent trong pipeline:

| Agent | Model | Tại sao chọn model này? |
|-------|-------|-------------------------|
| Research Agent | Claude Sonnet | Cần **tools** (DuckDuckGo). Sonnet nhanh và giá hợp lý. |
| Outline Agent | Claude Sonnet | Cần **output_schema** (JSON có cấu trúc). Sonnet xử lý được cả tools lẫn schema. |
| Writer Agent | Grok-4 | Viết tự nhiên, cuốn hút. Không cần tools hay schema — chỉ xuất Markdown thuần. |
| Image Agent | Claude Sonnet | Cần **cả** tools (API ảnh) lẫn output_schema. Chỉ Claude hỗ trợ kết hợp này. |
| Team Leader | Claude Opus | Suy luận phức tạp để hiểu yêu cầu người dùng và phân công cho đúng thành viên. |

**Ràng buộc then chốt:** Claude có thể kết hợp `tools` + `output_schema` trong cùng một agent. Grok **không thể**. Chỉ một hạn chế kỹ thuật này đã định hình toàn bộ kiến trúc của chúng ta. Writer Agent dùng Grok vì Grok viết rất tốt, và việc viết không cần tools hay schema.

In [ ]:
# Tại sao kiến trúc lại có hình dạng như vậy

print("=== Khả Năng Của Các Model ===\n")

capabilities = {
    "Claude Sonnet": {"tools": True, "output_schema": True, "both_together": True},
    "Claude Opus": {"tools": True, "output_schema": True, "both_together": True},
    "Grok-4": {"tools": True, "output_schema": True, "both_together": False},
}

for model, caps in capabilities.items():
    check = lambda x: "Yes" if x else "NO"
    print(f"  {model:15s}  tools: {check(caps['tools']):3s}  schema: {check(caps['output_schema']):3s}  kết hợp cả hai: {check(caps['both_together'])}")

print("\n=== Yêu Cầu Của Các Agent Trong Pipeline ===\n")

agents = [
    {"name": "Research Agent", "needs_tools": True, "needs_schema": False, "needs_both": False, "model": "Claude Sonnet"},
    {"name": "Outline Agent", "needs_tools": False, "needs_schema": True, "needs_both": False, "model": "Claude Sonnet"},
    {"name": "Writer Agent", "needs_tools": False, "needs_schema": False, "needs_both": False, "model": "Grok-4"},
    {"name": "Image Agent", "needs_tools": True, "needs_schema": True, "needs_both": True, "model": "Claude Sonnet"},
]

for agent in agents:
    needs = []
    if agent["needs_tools"]: needs.append("tools")
    if agent["needs_schema"]: needs.append("schema")
    if agent["needs_both"]: needs.append("kết hợp cả hai")
    needs_str = ", ".join(needs) if needs else "không cần gì"
    print(f"  {agent['name']:18s} cần: {needs_str:20s} \u2192 {agent['model']}")

print("\nWriter dùng Grok vì chỉ cần xuất văn bản thuần.")
print("Image Agent BẮT BUỘC dùng Claude vì cần tools + schema cùng lúc.")

## Embeddings — Một Dạng AI Khác

Không phải mọi output của AI đều là văn bản. **Embeddings** biến văn bản thành danh sách các con số thể hiện ý nghĩa.

Ví dụ: Tọa độ GPS. "Tokyo" và "Thủ đô Nhật Bản" nằm gần nhau trên bản đồ ý nghĩa. "Làm bánh mì" thì ở rất xa.

```
"SEO"                        → [0.82, 0.15, 0.91, ...]
"search engine optimization" → [0.80, 0.17, 0.89, ...]  ← gần nhau!
"baking bread"               → [0.12, 0.88, 0.05, ...]  ← rất xa
```

Embeddings được dùng ở đâu:
- **Tìm kiếm ngữ nghĩa** — tìm bài viết tương tự với truy vấn (không chỉ khớp từ khóa)
- **Phân cụm nội dung** — nhóm các bài viết tương tự lại với nhau
- **RAG** (Retrieval-Augmented Generation) — tìm tài liệu liên quan để đưa cho LLM

Chúng ta không dùng embeddings trong pipeline này, nhưng đây là khái niệm quan trọng cho các công cụ AI sau này.

In [ ]:
# Demo embedding đơn giản (embedding thật dùng hàng trăm chiều)
# Đây chỉ minh họa KHÁI NIỆM — số thật đến từ embedding model

fake_embeddings = {
    "SEO optimization": [0.9, 0.8, 0.1, 0.2],
    "search engine ranking": [0.85, 0.75, 0.15, 0.25],
    "content marketing": [0.6, 0.5, 0.7, 0.3],
    "baking sourdough bread": [0.1, 0.1, 0.2, 0.9],
}

def similarity(a, b):
    """Tích vô hướng đơn giản (hệ thống thật dùng cosine similarity)"""
    return sum(x * y for x, y in zip(a, b))

print("Điểm tương đồng (cao hơn = giống nhau hơn):\n")
base = "SEO optimization"
base_emb = fake_embeddings[base]

for text, emb in fake_embeddings.items():
    if text != base:
        score = similarity(base_emb, emb)
        bar = "\u2588" * int(score * 10)
        print(f"  \"{base}\" vs \"{text}\"")
        print(f"    Điểm: {score:.2f}  {bar}\n")

## Chọn Model Phù Hợp — Hướng Dẫn Quyết Định

Khi bạn cần quyết định dùng model nào cho một agent, hãy đi theo sơ đồ sau:

1. **Agent có cần tools không** (tìm kiếm web, API)?
   - Có → Phải dùng model hỗ trợ tools (Claude Sonnet/Opus)
2. **Agent có cần structured output không** (output_schema)?
   - Có → Phải dùng model hỗ trợ output_schema
3. **Có cần CẢ tools LẪN structured output không?**
   - Có → **Bắt buộc dùng Claude** (Grok không thể kết hợp cả hai)
   - Không → Linh hoạt hơn trong việc chọn model
4. **Tác vụ chủ yếu là viết văn dài?**
   - Có → Cân nhắc Grok (chất lượng viết rất tốt)
5. **Tác vụ yêu cầu suy luận phức tạp/phân công?**
   - Có → Cân nhắc Opus (mạnh nhất, nhưng đắt)
6. **Mặc định** → Claude Sonnet (cân bằng tốt nhất giữa tốc độ, chi phí và khả năng)

In [ ]:
# Bài tập: Ghép agent với model!
# Với mỗi tình huống, bạn sẽ chọn model nào và tại sao?

scenarios = [
    {
        "name": "Email Writer Agent",
        "description": "Viết email marketing từ brief",
        "needs_tools": False,
        "needs_schema": False,
        "priority": "Chất lượng viết",
    },
    {
        "name": "Competitor Analyzer",
        "description": "Tìm kiếm web và trả về dữ liệu đối thủ có cấu trúc",
        "needs_tools": True,
        "needs_schema": True,
        "priority": "Độ chính xác",
    },
    {
        "name": "Content Classifier",
        "description": "Phân loại bài viết theo chủ đề (trả về nhãn danh mục)",
        "needs_tools": False,
        "needs_schema": True,
        "priority": "Tốc độ và chi phí",
    },
]

print("=== Bài Tập Chọn Model ===\n")
for s in scenarios:
    print(f"Agent: {s['name']}")
    print(f"  Tác vụ: {s['description']}")
    print(f"  Cần tools: {'Có' if s['needs_tools'] else 'Không'}")
    print(f"  Cần schema: {'Có' if s['needs_schema'] else 'Không'}")
    print(f"  Ưu tiên: {s['priority']}")
    print(f"  Lựa chọn của bạn: _______________")
    print()

print("Hãy suy nghĩ câu trả lời, rồi xem đáp án gợi ý bên dưới!")

In [ ]:
# Đáp án gợi ý:

answers = {
    "Email Writer Agent": "Grok-4 \u2014 ưu tiên chất lượng viết, không cần tools hay schema",
    "Competitor Analyzer": "Claude Sonnet \u2014 cần cả tools LẪN schema cùng lúc, chỉ Claude hỗ trợ",
    "Content Classifier": "Claude Haiku \u2014 tác vụ đơn giản, cần schema nhưng nhẹ, ưu tiên tốc độ và chi phí",
}

print("=== Đáp Án Gợi Ý ===\n")
for agent, answer in answers.items():
    print(f"  {agent}: {answer}\n")

## Tổng Kết Mô-đun 2 — Hiểu Về AI

| Bài học | Chủ đề | Điểm mấu chốt |
|---------|--------|----------------|
| Bài học 5 | Cách LLM Hoạt Động | LLM dự đoán văn bản từng token một. Chúng có knowledge cutoff và có thể bịa thông tin (hallucinate). |
| Bài học 6 | Prompt và Context | Prompt tốt = Vai trò + Tác vụ + Ràng buộc + Ví dụ. `instructions` chính là system prompt. |
| Bài học 7 | Model và Cách Lựa Chọn | Mỗi model phù hợp với mỗi tác vụ khác nhau. Kiến trúc được định hình bởi khả năng của model. |

Bây giờ bạn đã hiểu **TẠI SAO** các agent hoạt động theo cách đó. Trong Mô-đun 3, bạn sẽ **XÂY DỰNG** chúng.

**Mô-đun tiếp theo:** Mô-đun 3 — Xây Dựng Agent. Bạn sẽ tạo agent AI đầu tiên, trang bị tools cho nó, khiến nó trả về dữ liệu có cấu trúc, nối chuỗi các agent lại, và xây dựng một mini pipeline!

## Bài Tập

Không cần code — bài tập tư duy:

Agency SEO của bạn muốn xây một công cụ AI mới: **"Content Audit Agent"** có khả năng:
- Crawl URL website để đọc nội dung hiện có (cần tool)
- Phân tích nội dung để tìm vấn đề SEO
- Trả về báo cáo có cấu trúc với điểm số và khuyến nghị (cần schema)
- Phải đủ rẻ để chạy trên hàng trăm trang

**Câu hỏi:**
1. Bạn sẽ chọn model nào? Tại sao?
2. Có thể dùng Grok cho việc này không? Tại sao có hoặc tại sao không?
3. Làm sao để giữ chi phí thấp khi xử lý hàng trăm trang?

<details>
<summary>Nhấn để xem đáp án gợi ý</summary>

1. **Claude Sonnet** — cần cả tools (crawl web) lẫn structured output (schema báo cáo), và Sonnet đủ rẻ để dùng hàng loạt.
2. **Không** — Grok không thể kết hợp tools và output_schema trong cùng một agent. Bạn sẽ phải tách thành hai agent riêng biệt (một cho crawl, một cho báo cáo), làm tăng độ phức tạp.
3. Dùng **Claude Haiku** cho bước lọc sơ bộ (kiểm tra nhanh xem trang có cần audit kỹ không), rồi chỉ gửi những trang cần phân tích sâu tới Sonnet. Ngoài ra, giữ prompt ngắn gọn để giảm thiểu input token.

</details>